In [ ]:
from ugot import ugot
import numpy as np
from IPython.display import display, clear_output, Image
import cv2

got = ugot.UGOT()
got.initialize("192.168.1.189")
got.load_models(["apriltag_qrcode"])

In [ ]:
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()
        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        tags = got.get_apriltag_total_info()
        if tags:
            tag = tags[0][0:6]  # Only look at the first detected tag
            cv2.putText(img, f"tag: {tag}", (20, 70), 0, 0.8, (0, 255, 0), 2)
        else:
            cv2.putText(img, "No tag!", (20, 70), 0, 0.8, (0, 0, 255), 2)
        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))
except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()

In [ ]:
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()
        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        tags = got.get_apriltag_total_info()
        if tags:
            tag = tags[0]  # Only look at the first detected tag

            # Extract center and dimensions
            cx, cy = int(tag[1]), int(tag[2])
            half_w, half_h = int(tag[4] / 2), int(tag[3] / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with tag ID and distance (using 5x5 cm reference)
            label = f"ID: {tag[0]}  dist: {tag[6]:.2f}cm"
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        else:
            cv2.putText(img, "No tag!", (20, 70), 0, 0.8, (0, 0, 255), 2)

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))
except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()